<a href="https://colab.research.google.com/github/sahanabytes/machine-learning-portfolio/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sahanabytes/machine-learning-portfolio/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding 1: The Anatomy of Growing Content**

The report compares content pages that are gaining traffic with pages that are losing traffic. It found that growing pages were younger on average (185 days) than declining pages (228 days). However, the average word count was almost the same in both groups. This suggests that content age may be associated with traffic growth, while word count alone may not explain the difference.

**My methodology question:**

How exactly were pages labelled as growing or declining? Which time periods and thresholds were used? Was this finding also checked using a time-aware or held-out validation method? Since other factors can affect traffic, does the result show an association rather than prove that younger content causes growth?

**Finding 2: The Freshness Multiplier**

The report examined the relationship between content freshness and traffic growth. It found that pages updated 31–90 days earlier had a growth-to-decline ratio of 5.43:1. However, the 361+ days group showed a much higher ratio of 37.19:1, but this group had only 802 pages and just 21 declining pages. Therefore, the result for this small group should be interpreted carefully.

**My methodology question:**

How were the growth and decline labels defined, and were the same time windows used for every freshness group? Was the result checked on separate or held-out data? Since the 361+ days group is relatively small, could a few pages have influenced the ratio? Does this finding show an association between freshness and traffic direction rather than prove that updating content causes traffic growth?

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

I will re-run my Week-5 Decision Tree model using a grouped split based on `client_id`. This ensures that each client's data appears in either the training set or the test set, but not both. I will compare the Week 4 baseline and the Decision Tree using Precision@20 and Precision@50 on the same held-out test data. These measurements evaluate how well the methods identify pages with observed declines. They do not prove that refreshing a page will improve its performance.

In [16]:

import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier

# Load the same dataset used in Week 5
data_url = (
    "https://raw.githubusercontent.com/"
    "sahanabytes/machine-learning-portfolio/"
    "main/data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(data_url)

# Define the observed decline label
df["decline_label"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

# Use the same grouped split as Week 5
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        df,
        y=df["decline_label"],
        groups=df["client_id"]
    )
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

# Confirm that clients do not overlap
assert set(train_df["client_id"]).isdisjoint(
    set(test_df["client_id"])
)

# Use the exact Week-5 feature set
feature_cols = [
    "days_since_last_update",
    "impressions_90d",
    "search_volume",
    "competition",
    "word_count",
    "content_age_days",
    "avg_position",
    "ctr"
]

X_train = train_df[feature_cols].apply(
    pd.to_numeric, errors="coerce"
)
X_test = test_df[feature_cols].apply(
    pd.to_numeric, errors="coerce"
)

# Fit missing-value replacements on training data only
train_medians = X_train.median()
X_train = X_train.fillna(train_medians).fillna(0)
X_test = X_test.fillna(train_medians).fillna(0)

y_train = train_df["decline_label"]
y_test = test_df["decline_label"]

# Train the same Decision Tree as Week 5
model = DecisionTreeClassifier(
    max_depth=3,
    min_samples_leaf=100,
    random_state=42
)

model.fit(X_train, y_train)

# Calculate Week-4 baseline scores
baseline_score = (
    (test_df["days_since_last_update"] >= 180).astype(int) * 50
    + (test_df["impressions_90d"] >= 500).astype(int) * 50
)

# Calculate Decision Tree scores
model_score = model.predict_proba(X_test)[:, 1]

# Precision at K
def precision_at_k(y_true, scores, k):
    top_indices = np.argsort(-np.asarray(scores))[:k]
    return np.asarray(y_true)[top_indices].mean()

# Compare both methods
results = pd.DataFrame([
    {
        "Method": "Week 4 baseline",
        "Precision@20": precision_at_k(y_test, baseline_score, 20),
        "Precision@50": precision_at_k(y_test, baseline_score, 50)
    },
    {
        "Method": "Decision Tree",
        "Precision@20": precision_at_k(y_test, model_score, 20),
        "Precision@50": precision_at_k(y_test, model_score, 50)
    }
])

print("Grouped validation results")
print("Training clients:", train_df["client_id"].nunique())
print("Testing clients:", test_df["client_id"].nunique())
print("Client overlap:", set(train_df["client_id"]) & set(test_df["client_id"]))

print("\nBefore/after model comparison:")
print(results.round(3).to_string(index=False))


Grouped validation results
Training clients: 24
Testing clients: 8
Client overlap: set()

Before/after model comparison:
         Method  Precision@20  Precision@50
Week 4 baseline          0.60          0.60
  Decision Tree          0.45          0.56


### Results and interpretation

The grouped validation used 24 clients for training and 8 separate clients for testing, with no client overlap.

The Week 4 baseline achieved Precision@20 of 0.60 and Precision@50 of 0.60. The Decision Tree achieved Precision@20 of 0.45 and Precision@50 of 0.56.

The Decision Tree did not outperform the baseline on either metric. This suggests that the simpler baseline was more effective at ranking pages with observed declines in this evaluation. The results are directional and support decision-making, but they do not prove that refreshing a page will improve its traffic.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Leakage audit

My target is `decline_label`, which is derived from `trend_direction`. Therefore, I must not use `trend_direction`, `trend_pct`, or `decline_label` as model features.

I also exclude identifiers such as `client_id` and `content_id` because they identify clients or content items rather than provide useful prediction features.

My final feature set contains only the eight selected content-related variables. I will check that none of the target or identifier columns are included.

In [17]:

# Final feature set used by the Decision Tree
feature_cols = [
    "days_since_last_update",
    "impressions_90d",
    "search_volume",
    "competition",
    "word_count",
    "content_age_days",
    "avg_position",
    "ctr"
]

# Columns that must not enter the model features
blocked_cols = [
    "decline_label",
    "trend_direction",
    "trend_pct",
    "client_id",
    "content_id"
]

print("Final model features:")
print(feature_cols)

print("\nLeakage and identifier audit:")

found_blocked = [
    col for col in feature_cols if col in blocked_cols
]

print("Blocked columns found in features:", found_blocked)

assert len(found_blocked) == 0, "Leakage detected!"

print("\nTarget column:", "decline_label")
print("Target included in features:", "decline_label" in feature_cols)

print("\nLeakage audit passed.")


Final model features:
['days_since_last_update', 'impressions_90d', 'search_volume', 'competition', 'word_count', 'content_age_days', 'avg_position', 'ctr']

Leakage and identifier audit:
Blocked columns found in features: []

Target column: decline_label
Target included in features: False

Leakage audit passed.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Claim rewrite

**Original claim:**

The Decision Tree can identify pages that are likely to decline and help improve their search performance.

**Rewritten claim:**

In this evaluation, the Decision Tree measured a Precision@20 of 0.45 and Precision@50 of 0.56 on held-out clients. The Week 4 baseline achieved higher values on both metrics. These results suggest that the baseline was more effective at ranking pages with observed declines in this experiment. The model can support content review decisions, but its predictions do not prove that refreshing a page will improve its search performance.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.